# CTM-video visualisations

Run after training (or just after a few hundred synthetic iterations). Point `CKPT_DIR` at a training log directory; the notebook loads the checkpoint, runs one clip through the model with `track=True`, and produces the interpretable plots described in the task README.

1. Clip preview (frames grid).
2. Per-tick certainty curve with frame boundaries.
3. Top-k class probabilities evolving across ticks.
4. Neuron raster — D neurons x ticks heatmap.
5. Per-frame attention heatmap overlaid on the video frames.
6. UMAP of the output-synchronisation vector across ticks.
7. Per-neuron FFT — do different neurons learn different temporal frequencies?
8. Synchronisation magnitude over time (the soft-infinite memory channel).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
# Allow running this notebook from tasks/video/analysis/ with repo root on path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

from tasks.video.model import ContinuousThoughtMachineVideo
from tasks.video.dataset import build_datasets

CKPT_DIR = 'logs/video/synthetic'   # <- edit me
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
ckpt = torch.load(f'{CKPT_DIR}/checkpoint.pt', map_location=DEVICE, weights_only=False)
args = ckpt['args']
class_labels = ckpt['class_labels']

_, test_data, _ = build_datasets(args.dataset, args.data_root, args.n_frames, args.image_size,
                                 fold=getattr(args, 'fold', 1))

model = ContinuousThoughtMachineVideo(
    n_frames=args.n_frames,
    iterations_per_frame=args.iterations_per_frame,
    d_model=args.d_model, d_input=args.d_input, heads=args.heads,
    n_synch_out=args.n_synch_out, n_synch_action=args.n_synch_action,
    synapse_depth=args.synapse_depth, memory_length=args.memory_length,
    deep_nlms=args.deep_memory, memory_hidden_dims=args.memory_hidden_dims,
    do_layernorm_nlm=args.do_normalisation,
    backbone_type=args.backbone_type,
    positional_embedding_type=args.positional_embedding_type,
    out_dims=args.out_dims, prediction_reshaper=[-1],
    dropout=0.0, neuron_select_type=args.neuron_select_type,
).to(DEVICE)

# Init lazy modules with a dummy forward, then load weights.
dummy, _ = test_data[0]
model(dummy.unsqueeze(0).to(DEVICE))
model.load_state_dict(ckpt['model_state_dict'])
model.eval();

In [ ]:
# Pick a clip and run forward with tracking on.
CLIP_INDEX = 0
clip, label = test_data[CLIP_INDEX]
clip_t = clip.unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    (predictions, certainties, synch_tracks,
     pre_acts, post_acts, attention, frame_idx_track, kv_shape) = model(clip_t, track=True)

predictions = predictions.cpu().numpy()[0]          # (C, ticks)
certainties = certainties.cpu().numpy()[0]          # (2, ticks)
synch_out_track, synch_action_track = synch_tracks   # (ticks, B, S)
post_acts = post_acts[:, 0]                          # (ticks, D)
attention = attention[:, 0]                          # (ticks, heads, 1, tokens) or similar

print('true label :', class_labels[label])
print('final pred :', class_labels[predictions[:, -1].argmax()])
print('most-certain tick pred:', class_labels[predictions[:, certainties[1].argmax()].argmax()])
print('attention shape:', attention.shape, ' kv_shape:', kv_shape, ' n_frames:', args.n_frames)

## 1. Clip preview

In [ ]:
def denorm(clip_np):
    # synthetic data is already in [-1, 1]; real clips were imagenet-normalised.
    if args.dataset == 'synthetic':
        return np.clip(clip_np * 0.5 + 0.5, 0, 1)
    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
    return np.clip(clip_np * std + mean, 0, 1)

clip_np = denorm(clip.numpy())  # (T, 3, H, W)
T = clip_np.shape[0]
cols = min(8, T); rows = int(np.ceil(T / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
axes = np.array(axes).reshape(rows, cols)
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]; ax.axis('off')
    if i < T: ax.imshow(np.moveaxis(clip_np[i], 0, -1)); ax.set_title(f'f{i}', fontsize=8)
fig.suptitle(f'Clip preview ({class_labels[label]})', fontsize=12); fig.tight_layout(); plt.show()

## 2. Certainty over ticks

`certainties[1]` is `1 - normalised_entropy(predictions_t)`. Green bands mark ticks where the argmax equals the true label.

In [ ]:
ticks = np.arange(certainties.shape[1])
is_correct = (predictions.argmax(0) == label)
fig, ax = plt.subplots(figsize=(12, 3))
for t in ticks:
    ax.axvspan(t, t + 1, color='limegreen' if is_correct[t] else 'lightcoral', alpha=0.15)
ax.plot(ticks, certainties[1], 'k-', lw=1.5)
# Mark frame boundaries (every iterations_per_frame ticks).
for b in range(0, len(ticks), args.iterations_per_frame):
    ax.axvline(b - 0.5, color='steelblue', lw=0.4, alpha=0.4)
ax.set_ylim(-0.02, 1.02); ax.set_xlim(0, len(ticks))
ax.set_xlabel('internal tick'); ax.set_ylabel('1 - normalised entropy')
ax.set_title('Per-tick certainty (green = correct argmax)'); fig.tight_layout(); plt.show()

## 3. Top-k class probabilities across ticks

In [ ]:
probs = torch.softmax(torch.from_numpy(predictions), dim=0).numpy()  # (C, ticks)
TOPK = min(6, probs.shape[0])
avg_prob = probs.mean(axis=1)
top_idx = np.argsort(avg_prob)[::-1][:TOPK]
fig, ax = plt.subplots(figsize=(12, 4))
for ci in top_idx:
    color = 'green' if ci == label else None
    ax.plot(probs[ci], label=class_labels[ci], color=color, lw=1.5 if ci == label else 1.0)
ax.set_xlabel('internal tick'); ax.set_ylabel('p(class)'); ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=8); ax.set_title('Top-k class probability vs tick')
fig.tight_layout(); plt.show()

## 4. Neuron raster

Rows = a random subset of CTM neurons, columns = ticks. The thing to look for: different neurons develop different temporal profiles (oscillations, bursts, event-locked activations) rather than all following the same curve.

In [ ]:
N_NEURONS = 64
# Sort neurons by dominant frequency so visual structure emerges.
fft_mag = np.abs(np.fft.rfft(post_acts, axis=0))[1:]   # skip DC
peak_freq = fft_mag.argmax(axis=0)
order = np.argsort(peak_freq)
chosen = order[np.linspace(0, len(order) - 1, N_NEURONS).astype(int)]
raster = post_acts[:, chosen].T  # (N_NEURONS, ticks)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(raster, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(raster).max(), vmax=np.abs(raster).max(), interpolation='nearest')
ax.set_xlabel('internal tick'); ax.set_ylabel('neuron (sorted by dominant frequency)')
ax.set_title('Per-neuron post-activation raster')
plt.colorbar(im, ax=ax, shrink=0.7); fig.tight_layout(); plt.show()

## 5. Attention overlay per frame

At each tick, cross-attention produced weights over the spatial tokens of the *current* frame (as routed by `stepi // iterations_per_frame`). Here we reshape those weights back to (H', W'), upsample them to the image size, and overlay onto each frame.

In [ ]:
import torch.nn.functional as F

Hp, Wp = kv_shape
# `attention` shape: (ticks, heads, 1, Hp*Wp). Mean over heads, drop the singleton query dim.
attn = attention.mean(axis=1)
attn = attn.reshape(attn.shape[0], 1, Hp, Wp)  # (ticks, 1, Hp, Wp)
attn_tensor = torch.from_numpy(attn).float()
H = clip_np.shape[2]; W = clip_np.shape[3]
attn_up = F.interpolate(attn_tensor, size=(H, W), mode='bilinear', align_corners=False).squeeze(1).numpy()

# One subplot per sampled frame; show attention averaged across the K ticks
# that attended to that frame.
K = args.iterations_per_frame
per_frame_attn = np.stack([attn_up[f * K:(f + 1) * K].mean(0) for f in range(T)], axis=0)

cols = min(8, T); rows = int(np.ceil(T / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
axes = np.array(axes).reshape(rows, cols)
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]; ax.axis('off')
    if i < T:
        img = np.moveaxis(clip_np[i], 0, -1)
        ax.imshow(img); a = per_frame_attn[i]
        a = (a - a.min()) / (a.max() - a.min() + 1e-9)
        ax.imshow(a, cmap='hot', alpha=0.5); ax.set_title(f'frame {i}', fontsize=8)
fig.suptitle('Attention overlay per frame', fontsize=12); fig.tight_layout(); plt.show()

## 6. UMAP of the output-synchronisation vector across ticks

The output-sync is the representation that actually drives the classifier. Plotting its trajectory through time shows how the latent evolves: expect clusters near early frames and movement towards a class-specific region as the model gathers evidence.

In [ ]:
try:
    import umap
    sync = synch_out_track[:, 0]  # (ticks, S)
    reducer = umap.UMAP(n_components=2, n_neighbors=min(15, max(2, sync.shape[0] - 1)),
                        min_dist=0.6, metric='cosine', random_state=0)
    emb = reducer.fit_transform(sync)
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.cm.viridis
    for i in range(len(emb) - 1):
        ax.plot(emb[i:i+2, 0], emb[i:i+2, 1], '-', color=cmap(i / max(1, len(emb) - 1)), lw=1.5)
    ax.scatter(emb[:, 0], emb[:, 1], c=np.arange(len(emb)), cmap='viridis', s=40, edgecolor='k', lw=0.5)
    ax.scatter(emb[0, 0], emb[0, 1], c='white', edgecolor='black', s=120, label='start', zorder=5)
    ax.scatter(emb[-1, 0], emb[-1, 1], c='red', edgecolor='black', s=120, label='end', zorder=5)
    ax.legend(); ax.set_title('Output-synchronisation trajectory (UMAP, colour = tick)')
    fig.tight_layout(); plt.show()
except Exception as e:
    print('UMAP unavailable or failed, falling back to PCA:', e)
    from sklearn.decomposition import PCA
    sync = synch_out_track[:, 0]
    emb = PCA(n_components=2).fit_transform(sync)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(emb[:, 0], emb[:, 1], 'k-', alpha=0.5)
    ax.scatter(emb[:, 0], emb[:, 1], c=np.arange(len(emb)), cmap='viridis', s=40, edgecolor='k', lw=0.5)
    ax.set_title('Output-synchronisation trajectory (PCA)'); fig.tight_layout(); plt.show()

## 7. Per-neuron FFT

If the NLMs are doing their job, different neurons should respond at different temporal frequencies (oscillators, ramp neurons, event detectors). This is a key claim of the CTM paper — here you check whether it holds on your trained video model.

In [ ]:
fft_all = np.abs(np.fft.rfft(post_acts, axis=0))  # (freq, D)
fft_all = fft_all / (fft_all.sum(axis=0, keepdims=True) + 1e-9)
neuron_order = fft_all[1:].argmax(axis=0).argsort()
fft_sorted = fft_all[:, neuron_order]
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(fft_sorted[1:].T, aspect='auto', cmap='magma')
ax.set_xlabel('frequency bin'); ax.set_ylabel('neuron (sorted by peak freq)')
ax.set_title('FFT magnitude per neuron (each row normalised)')
plt.colorbar(im, ax=ax, shrink=0.7); fig.tight_layout(); plt.show()

## 8. Synchronisation magnitude over time

Two sync tracks — action (drives attention) and output (drives predictions). The RMS over the sync vector per tick is a crude view of the "soft-infinite" memory channel: how much the learned leaky integrators have accumulated.

In [ ]:
sa = np.sqrt((synch_action_track[:, 0] ** 2).mean(axis=-1))
so = np.sqrt((synch_out_track[:, 0] ** 2).mean(axis=-1))
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sa, label='action sync RMS', color='tab:orange')
ax.plot(so, label='output sync RMS', color='tab:blue')
for b in range(0, len(sa), args.iterations_per_frame):
    ax.axvline(b - 0.5, color='grey', lw=0.4, alpha=0.4)
ax.set_xlabel('internal tick'); ax.set_ylabel('||sync||_rms')
ax.set_title('Synchronisation accumulator magnitude'); ax.legend()
fig.tight_layout(); plt.show()